# Deploy Demo Application (Cafe Order System)

This notebook deploys the **Cafe Order System** — a sample internal enterprise application used to demonstrate AI-powered code search and documentation Q&A via MCP servers.

## What is the Cafe Order System?

A FastAPI-based REST API for managing a corporate cafeteria:
- **Menu management** (15+ items across coffee, tea, bakery, etc.)
- **Customer registry** (employee ID-based)
- **Order processing** (status flow: pending → confirmed → preparing → ready → picked_up)

This app serves as the **target codebase** for the custom MCP servers:
- `mcp-codebase-search` indexes its Python source code
- `mcp-repo-docs` indexes its internal documentation

## 1. Review Application Structure

In [1]:
%%bash
echo "=== Cafe Order System — File Structure ==="
echo ""
find apps/cafe-order-system -type f | sort | head -30
echo ""
echo "=== Source code ($(find apps/cafe-order-system/app -name '*.py' | wc -l) Python files) ==="
find apps/cafe-order-system/app -name '*.py' | sort
echo ""
echo "=== Documentation ($(find apps/cafe-order-system/docs -name '*.md' | wc -l) docs) ==="
find apps/cafe-order-system/docs -name '*.md' | sort

=== Cafe Order System — File Structure ===

apps/cafe-order-system/Dockerfile
apps/cafe-order-system/app/__init__.py
apps/cafe-order-system/app/config.py
apps/cafe-order-system/app/database.py
apps/cafe-order-system/app/main.py
apps/cafe-order-system/app/models.py
apps/cafe-order-system/app/routes/__init__.py
apps/cafe-order-system/app/routes/customers.py
apps/cafe-order-system/app/routes/menu.py
apps/cafe-order-system/app/routes/orders.py
apps/cafe-order-system/app/schemas.py
apps/cafe-order-system/app/services/__init__.py
apps/cafe-order-system/app/services/inventory_service.py
apps/cafe-order-system/app/services/order_service.py
apps/cafe-order-system/docs/api-guide.md
apps/cafe-order-system/docs/architecture.md
apps/cafe-order-system/docs/onboarding.md
apps/cafe-order-system/docs/runbook.md
apps/cafe-order-system/docs/security-policy.md
apps/cafe-order-system/requirements.txt

=== Source code (      13 Python files) ===
apps/cafe-order-system/app/__init__.py
apps/cafe-order-system/

## 2. Quick Look at Key Files

In [2]:
%%bash
echo "=== Data Model (app/models.py) — Key Classes ==="
echo ""
grep -n "^class " apps/cafe-order-system/app/models.py
echo ""
echo "=== API Routes ==="
echo ""
grep -n "@router" apps/cafe-order-system/app/routes/*.py | sed 's|apps/cafe-order-system/app/||'
echo ""
echo "=== Documentation Topics ==="
echo ""
for doc in apps/cafe-order-system/docs/*.md; do
    title=$(head -1 "$doc" | sed 's/# //')
    printf "  %-25s %s\n" "$(basename $doc)" "$title"
done

=== Data Model (app/models.py) — Key Classes ===

10:class MenuCategory(str, PyEnum):
18:class OrderStatus(str, PyEnum):
27:class MenuItem(Base):
43:class Customer(Base):
57:class Order(Base):
73:class OrderItem(Base):

=== API Routes ===

routes/customers.py:11:@router.get("/", response_model=list[CustomerResponse])
routes/customers.py:22:@router.get("/{employee_id}", response_model=CustomerResponse)
routes/customers.py:32:@router.post("/", response_model=CustomerResponse, status_code=201)
routes/customers.py:48:@router.get("/{employee_id}/orders")
routes/menu.py:11:@router.get("/", response_model=list[MenuItemResponse])
routes/menu.py:25:@router.get("/{item_id}", response_model=MenuItemResponse)
routes/menu.py:33:@router.post("/", response_model=MenuItemResponse, status_code=201)
routes/menu.py:42:@router.patch("/{item_id}/availability")
routes/menu.py:52:@router.get("/search/", response_model=list[MenuItemResponse])
routes/orders.py:13:@router.get("/", response_model=list[OrderRespo

## 3. Deploy to OpenShift (Optional)

Deploy the Cafe Order System as a running service. This is optional — the MCP servers only need the **source code files** (mounted via ConfigMap), not a running instance.

However, deploying it demonstrates a realistic internal service.

In [3]:
%%bash
echo "Creating namespace..."
oc create namespace cafe-system 2>/dev/null || echo "(namespace exists)"

echo ""
echo "Building container image..."
if ! oc get bc cafe-api -n cafe-system &>/dev/null; then
    oc new-build --binary --strategy=docker --name=cafe-api -n cafe-system
fi

oc start-build cafe-api \
    --from-dir=apps/cafe-order-system \
    -n cafe-system \
    --follow --wait

Creating namespace...
namespace/cafe-system created

Building container image...
    * A Docker build using binary input will be created
      * The resulting image will be pushed to image stream tag "cafe-api:latest"
      * A binary build was created, use 'oc start-build --from-dir' to trigger a new build

--> Creating resources with label build=cafe-api ...
    imagestream.image.openshift.io "cafe-api" created
    buildconfig.build.openshift.io "cafe-api" created
--> Success


Uploading directory "apps/cafe-order-system" as binary input for the build ...
...
Uploading finished


build.build.openshift.io/cafe-api-1 started
Receiving source from STDIN as archive ...
time="2026-06-18T08:30:51Z" level=info msg="Not using native diff for overlay, this may cause degraded performance for building images: kernel has CONFIG_OVERLAY_FS_REDIRECT_DIR enabled"
I0618 08:30:51.758408       1 defaults.go:112] Defaulting to storage driver "overlay" with options [mountopt=metacopy=on].
Caching blobs under "/var/cache/blobs".

Pulling image python:3.11-slim ...
Resolving "python" using unqualified-search registries (/etc/containers/registries.conf)
Trying to pull registry.redhat.io/python:3.11-slim...
Trying to pull registry.access.redhat.com/python:3.11-slim...
Trying to pull quay.io/python:3.11-slim...
Trying to pull docker.io/library/python:3.11-slim...
Getting image source signatures
Copying blob sha256:3250ea7ceadc1e9a40a2d832082ee8fe68631ef9c9bf7bd5447fbcff9f527439
Copying blob sha256:72c03230f1363a3fb61d2f98504cf168bad3fe22f511ad2005dc021515d7ce97
Copying blob sha256:8c47

In [4]:
%%bash
echo "Deploying cafe-api..."

# Create deployment if not exists
if ! oc get deploy cafe-api -n cafe-system &>/dev/null; then
    oc create deployment cafe-api \
        --image=image-registry.openshift-image-registry.svc:5000/cafe-system/cafe-api:latest \
        -n cafe-system
    oc set resources deploy/cafe-api -n cafe-system --requests=cpu=100m,memory=128Mi --limits=cpu=500m,memory=256Mi
    oc expose deploy/cafe-api --port=8000 -n cafe-system
    oc expose svc/cafe-api -n cafe-system
fi

# Wait for rollout
oc rollout restart deploy/cafe-api -n cafe-system
oc wait --for=condition=available deployment/cafe-api -n cafe-system --timeout=120s

echo ""
echo "Cafe API endpoint:"
ROUTE=$(oc get route cafe-api -n cafe-system -o jsonpath='{.spec.host}' 2>/dev/null)
echo "  http://${ROUTE}"
echo ""
echo "Health check:"
HEALTH=$(curl -s --max-time 5 "http://${ROUTE}/health" 2>/dev/null)
if [ -n "$HEALTH" ]; then
    echo "$HEALTH" | python3 -m json.tool 2>/dev/null || echo "$HEALTH"
else
    echo "⚠️  No response yet — pod may still be starting. Retry in a few seconds."
fi

Deploying cafe-api...
deployment.apps/cafe-api created
deployment.apps/cafe-api resource requirements updated
service/cafe-api exposed
route.route.openshift.io/cafe-api exposed
deployment.apps/cafe-api restarted
deployment.apps/cafe-api condition met

Cafe API endpoint:
  http://cafe-api-cafe-system.apps.openshift-cluster.sandbox1785.opentlc.com

Health check:


Expecting value: line 1 column 1 (char 0)


CalledProcessError: Command 'b'echo "Deploying cafe-api..."\n\n# Create deployment if not exists\nif ! oc get deploy cafe-api -n cafe-system &>/dev/null; then\n    oc create deployment cafe-api \\\n        --image=image-registry.openshift-image-registry.svc:5000/cafe-system/cafe-api:latest \\\n        -n cafe-system\n    oc set resources deploy/cafe-api -n cafe-system --requests=cpu=100m,memory=128Mi --limits=cpu=500m,memory=256Mi\n    oc expose deploy/cafe-api --port=8000 -n cafe-system\n    oc expose svc/cafe-api -n cafe-system\nfi\n\n# Wait for rollout\noc rollout restart deploy/cafe-api -n cafe-system\noc wait --for=condition=available deployment/cafe-api -n cafe-system --timeout=120s\n\necho ""\necho "Cafe API endpoint:"\nROUTE=$(oc get route cafe-api -n cafe-system -o jsonpath=\'{.spec.host}\' 2>/dev/null)\necho "  http://${ROUTE}"\necho ""\necho "Health check:"\ncurl -s "http://${ROUTE}/health" | python3 -m json.tool\n'' returned non-zero exit status 1.

## 4. Verify API Functionality

In [ ]:
%%bash
ROUTE=$(oc get route cafe-api -n cafe-system -o jsonpath='{.spec.host}' 2>/dev/null)
BASE="http://${ROUTE}"

echo "=== Menu (coffee category) ==="
curl -s "${BASE}/api/menu/?category=coffee" | python3 -m json.tool | head -30

echo ""
echo "=== Customers ==="
curl -s "${BASE}/api/customers/" | python3 -m json.tool | head -20

echo ""
echo "=== Create Order ==="
curl -s -X POST "${BASE}/api/orders/" \
    -H "Content-Type: application/json" \
    -d '{"customer_id": 1, "items": [{"menu_item_id": 1, "quantity": 2, "customization": "ice"}], "notes": "test order"}' \
    | python3 -m json.tool

## Summary

The Cafe Order System is now deployed. Its source code and documentation will be indexed by the custom MCP servers in the next phase.

| Component | Location | Purpose |
|-----------|----------|--------|
| Source code | `apps/cafe-order-system/app/` | Indexed by `mcp-codebase-search` |
| Documentation | `apps/cafe-order-system/docs/` | Indexed by `mcp-repo-docs` |
| Running API | `http://cafe-api-cafe-system.apps.CLUSTER/` | Demo target |

## Next Steps

→ `../1_mcp_servers/2_deploy_mcp_servers.ipynb` — Deploy MCP servers that index this app's code and docs